# 选做实验：Tree Ensembles

本 Notebook 将练习：

- 使用 Pandas 对数据集进行 One-hot Encoding。
- 使用 Scikit-learn 实现 Decision Tree、Random Forest，并使用 XGBoost 构建 Gradient Boosting 模型。


首先导入所需的库。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
!pip install xgboost --quiet
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

RANDOM_STATE = 55 ## You will pass it to every sklearn call so we ensure reproducibility

## 1. 加载数据集


数据来自 [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)。

**数据背景**

原始数据集介绍指出，Cardiovascular Diseases（CVDs，心血管疾病）是全球主要死因，每年估计造成 1,790 万人死亡，约占全球死亡人数的 31%。Heart Failure（心力衰竭）是 CVDs 可能引起的常见后果。该数据集包含 11 个可用于预测 Heart Disease 的特征。

患有 CVDs 或处于高风险的人群需要早期发现与管理，Machine Learning 模型可为此类研究提供帮助。本实验将根据以下信息构建模型，预测样本中的 Heart Disease 标签。

**属性说明**

- `Age`：患者年龄，单位为年。
- `Sex`：性别，`M` 表示 Male，`F` 表示 Female。
- `ChestPainType`：胸痛类型，`TA` 为 Typical Angina，`ATA` 为 Atypical Angina，`NAP` 为 Non-Anginal Pain，`ASY` 为 Asymptomatic。
- `RestingBP`：静息血压，单位为 mm Hg。
- `Cholesterol`：血清胆固醇，单位为 mg/dl。
- `FastingBS`：空腹血糖是否超过 120 mg/dl；超过为 1，否则为 0。
- `RestingECG`：静息 ECG 结果。`Normal` 表示正常；`ST` 表示 ST-T 波异常，包括 T 波倒置和/或超过 0.05 mV 的 ST 段抬高或压低；`LVH` 表示根据 Estes 标准可能或明确存在 Left Ventricular Hypertrophy。
- `MaxHR`：达到的最大心率，数值在 60–202 之间。
- `ExerciseAngina`：是否有运动诱发的 Angina，`Y` 为是，`N` 为否。
- `Oldpeak`：ST 段压低的测量值。
- `ST_Slope`：运动峰值时 ST 段的斜率，`Up` 为上升，`Flat` 为平坦，`Down` 为下降。
- `HeartDisease`：输出类别，1 表示 Heart Disease，0 表示正常。


下面加载数据集。`Sex`、`ChestPainType`、`RestingECG`、`ExerciseAngina` 和 `ST_Slope` 是 Categorical Variables，需要进行编码。


In [ ]:
# Load the dataset using pandas
df = pd.read_csv("heart.csv")

In [ ]:
df.head()

训练模型前先进行数据处理。这里使用 Pandas 对上述五个 Categorical Features 进行 One-hot Encoding。


## 2. 使用 Pandas 进行 One-hot Encoding

先列出需要编码的 Categorical Variables。下面的实现将对五列全部进行 One-hot Encoding，其中也包括只有两个类别的 `Sex` 和 `ExerciseAngina`。


In [ ]:
cat_variables = ['Sex',
'ChestPainType',
'RestingECG',
'ExerciseAngina',
'ST_Slope'
]

One-hot Encoding 将一个具有 `n` 个类别的 Categorical Variable 转换为 `n` 个 Binary Variables。

Pandas 的 `pd.get_dummies` 提供了此功能。本实验使用以下参数：

- `data`：待处理的 DataFrame。
- `prefix`：输出列的前缀列表，便于识别各列来自哪个变量。
- `columns`：需要编码的列名列表，长度应与 `prefix` 一致。

可运行 `help(pd.get_dummies)` 查看完整文档。


In [ ]:
# This will replace the columns with the one-hot encoded ones and keep the columns outside 'columns' argument as it is.
df = pd.get_dummies(data = df,
                         prefix = cat_variables,
                         columns = cat_variables)

In [ ]:
df.head()

下面确定本实验模型最终使用的输入特征集合。


In [ ]:
var = [x for x in df.columns if x not in 'HeartDisease'] ## Removing our target variable

留意特征数量的变化。最初有 11 个输入变量，编码后数量如下：


In [ ]:
print(len(var))

## 3. 划分数据集

本节使用 Scikit-learn 的 `train_test_split`，将数据划分为 Training Set 和 Test Set。先查看该函数的参数。


In [ ]:
help(train_test_split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[var], df['HeartDisease'], train_size = 0.8, random_state = RANDOM_STATE)

# We will keep the shuffle = True since our dataset has not any time dependency.

In [ ]:
print(f'train samples: {len(X_train)}\ntest samples: {len(X_test)}')
print(f'target proportion: {sum(y_train)/len(y_train):.4f}')

## 4. 构建模型

### 4.1 Decision Tree

下面使用之前学习的 Decision Tree，这次采用 [Scikit-learn 的实现](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)。

`DecisionTreeClassifier` 提供多种 Hyperparameters。本实验仅逐项探索其中一部分，不进行 Feature Selection 或完整的参数组合搜索。你可以自行扩展并比较结果。

- `min_samples_split`：分裂一个内部节点所需的最少样本数，可用于减轻 Overfitting。
- `max_depth`：树的最大深度，也可用于减轻 Overfitting。


In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700] ## If the number is an integer, then it is the actual quantity of samples,
max_depth_list = [1,2, 3, 4, 8, 16, 32, 64, None] # None means that there is no depth limit.

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

可以看到，增大 `min_samples_split` 能减轻本例中的 Overfitting。下面对 `max_depth` 做同样的实验。


In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

本例中，`max_depth=3` 时 Test Accuracy 最高。允许的深度更小时，树无法进行足够多的分裂以区分正负样本，出现 Underfitting；深度过大时，树会过度适应 Training Set，导致 Test Accuracy 下降，即 Overfitting。最终使用：

- `max_depth=3`
- `min_samples_split=50`


In [ ]:
decision_tree_model = DecisionTreeClassifier(min_samples_split = 50,
                                             max_depth = 3,
                                             random_state = RANDOM_STATE).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_test),y_test):.4f}")

虽然指标还不够理想，但训练与测试表现之间没有明显的 Overfitting 迹象。


### 4.2 Random Forest

接下来使用 Scikit-learn 的 Random Forest。它由多个 Decision Trees 组成，因此也具有上述 Hyperparameters，另外还有 `n_estimators`，用于指定要训练的树的数量。

回顾 Random Forest：各棵树使用随机抽样的训练样本，而每次节点分裂只考虑随机选择的一部分特征。本例采用课程中介绍的 $\sqrt{n}$ 个候选特征，其中 $n$ 为总特征数；这个设置可以修改。运行 `help(RandomForestClassifier)` 可查看更多参数说明。

`n_jobs` 控制并行计算。各棵树可独立训练，因此增大并行任务数可能加快训练，而不改变模型设定。使用过多 CPU 核心也可能影响电脑其他任务的响应速度。

下面重复前面的比较流程，并额外尝试 `n_estimators` 为 10、50、100 的情况；本实验使用的默认值为 100。


In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700]  ## If the number is an integer, then it is the actual quantity of samples,
                                             ## If it is a float, then it is the percentage of the dataset
max_depth_list = [2, 4, 8, 16, 32, 64, None]
n_estimators_list = [10,50,100,500]

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list) 
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for n_estimators in n_estimators_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(n_estimators = n_estimators,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(n_estimators_list )),labels=n_estimators_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

下面使用以下参数拟合 Random Forest：

- `max_depth=8`
- `min_samples_split=10`
- `n_estimators=100`


In [ ]:
random_forest_model = RandomForestClassifier(n_estimators = 100,
                                             max_depth = 8, 
                                             min_samples_split = 10).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_test),y_test):.4f}")

上面展示了逐个搜索 Hyperparameter 的方法，但每次调整一个参数时，其余参数都固定在默认值，因此结果只反映当前固定组合下的效果。

如果三个 Hyperparameters 各有四个候选值，完整搜索需要比较 $4\times4\times4=64$ 种组合，而逐项尝试只会得到 $4+4+4=12$ 次结果。Scikit-learn 的 `GridSearchCV` 可以遍历参数组合，并通过 `refit` 自动使用最佳组合重新拟合模型。详见其[文档](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)。


### 4.3 XGBoost

最后尝试 Gradient Boosting 的一种实现：XGBoost。Boosting 依次训练多棵树，让后续树改进当前 Ensemble 的误差。

除控制单棵树的参数外，XGBoost 还包含 Learning Rate 等参数，用于控制每棵新树对模型更新的贡献。

训练时可以传入由 `(X_val, y_val)` 组成的评估数据列表。每一轮都会计算评估指标；如果连续 `early_stopping_rounds` 轮没有改善，就停止训练。这样可以自动判断合适的迭代轮数，并减轻树过多带来的 Overfitting。

首先从 Training Set 中留出一部分作为评估数据；这里不应使用 Test Set。


In [ ]:
n = int(len(X_train)*0.8) ## Let's use 80% to train and 20% to eval

In [ ]:
X_train_fit, X_train_eval, y_train_fit, y_train_eval = X_train[:n], X_train[n:], y_train[:n], y_train[n:]

可以将 `n_estimators` 设得较大，因为 Early Stopping 会在评估指标持续不再改善时终止训练。


In [ ]:
xgb_model = XGBClassifier(n_estimators = 500, learning_rate = 0.1,verbosity = 1, random_state = RANDOM_STATE)
xgb_model.fit(X_train_fit,y_train_fit, eval_set = [(X_train_eval,y_train_eval)], early_stopping_rounds = 50)
# Here we must pass a list to the eval_set, because you can have several different tuples ov eval sets. The parameter 
# early_stopping_rounds is the number of iterations that it will wait to check if the cost function decreased or not.
# If not, it will stop and get the iteration that returned the lowest metric on the eval set.

本例虽然将 `n_estimators` 设为 500，但由于评估集上的 Log Loss 不再改善，训练提前结束。仔细查看输出，最低 Log Loss 对应的迭代早于停止时刻；`best_iteration` 返回最佳迭代的索引，从 0 开始计数。下面查看它，以确定最佳预测对应的树的数量。


In [ ]:
xgb_model.best_iteration

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_test),y_test):.4f}")

本次实验中，Random Forest 的 Accuracy 最高，但各模型整体接近。XGBoost 未经过 Hyperparameter 搜索，也取得了与 Random Forest 接近的 Test Accuracy。XGBoost 提供了较多可调参数，进一步调优可能改善结果；训练速度则取决于数据、参数和运行环境。

恭喜！你已经学会使用 Scikit-learn 的 Decision Tree、Random Forest，以及 XGBoost。
